# Bronze — Olist Marketing Funnel e IPCA

Executar no Jupyter da stack do grupo. Colocar os três CSV no diretório de trabalho do Jupyter ou adaptar os caminhos locais. Esta camada conserva todas as datas, os nomes e valores originais, acrescentando `_data_ingestao` e `_ficheiro_origem`. As tabelas são Delta e registadas na base `bronze` do Hive Metastore. Reexecução = substituição integral das três tabelas deste notebook.

In [ ]:
from pathlib import Path
from hdfs import InsecureClient
FICHEIROS_LOCAIS = {
    "mql": "./olist_marketing_qualified_leads_dataset.csv",
    "closed_deals": "./olist_closed_deals_dataset.csv",
    "ipca": "./IBGE_IPCA.csv",
}
HDFS_ORIGEM = "/projeto/bronze/bruno"
for nome, ficheiro in FICHEIROS_LOCAIS.items():
    assert Path(ficheiro).is_file(), f"Ficheiro não encontrado: {ficheiro}"
client = InsecureClient("http://hdfs-nn:9870", user="anonymous")
for nome, ficheiro in FICHEIROS_LOCAIS.items():
    destino = f"{HDFS_ORIGEM}/{Path(ficheiro).name}"
    client.delete(destino)
    client.upload(destino, ficheiro, overwrite=True)
    print(nome, "->", destino, client.status(destino)["length"], "bytes")

In [ ]:
from pyspark.sql import SparkSession, functions as F
from delta import configure_spark_with_delta_pip
HDFS = "hdfs://hdfs-nn:9000"
WAREHOUSE = f"{HDFS}/warehouse"
builder = (SparkSession.builder.appName("EDSTD Bruno Marketing")
    .config("spark.sql.warehouse.dir", WAREHOUSE)
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0")
    .enableHiveSupport())
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)

In [ ]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS bronze LOCATION '{WAREHOUSE}/bronze.db/'")
def ler_csv(nome):
    caminho = f"{HDFS}{HDFS_ORIGEM}/{Path(FICHEIROS_LOCAIS[nome]).name}"
    return (spark.read.option("header", "true").option("inferSchema", "false")
            .option("multiLine", "true").option("escape", '"').csv(caminho))
def gravar_bronze(nome, tabela):
    df = (ler_csv(nome)
          .withColumn("_ficheiro_origem", F.input_file_name())
          .withColumn("_data_ingestao", F.current_timestamp()))
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .option("path", f"{WAREHOUSE}/bronze.db/{tabela}/")
       .saveAsTable(f"bronze.{tabela}"))
    n = spark.table(f"bronze.{tabela}").count()
    print(f"bronze.{tabela}: {n} linhas")
    return n
contagens = {
    "mql": gravar_bronze("mql", "bronze_mql"),
    "closed_deals": gravar_bronze("closed_deals", "bronze_closed_deals"),
    "ipca": gravar_bronze("ipca", "bronze_ipca"),
}

## Validação — contagens anteriores apenas como referência

In [ ]:
esperado = {"mql": 8000, "closed_deals": 842, "ipca": 393}
for nome, n in contagens.items():
    print(f"{nome}: {n} | referência: {esperado[nome]}")
    assert n == esperado[nome], f"Contagem diferente para {nome}; verificar origem"
for tabela in ("bronze_mql", "bronze_closed_deals", "bronze_ipca"):
    df = spark.table(f"bronze.{tabela}")
    assert {"_data_ingestao", "_ficheiro_origem"}.issubset(df.columns)
    assert df.filter(F.col("_data_ingestao").isNull() | F.col("_ficheiro_origem").isNull()).count() == 0
    print(tabela, df.columns)
spark.sql("SHOW TABLES IN bronze").show(truncate=False)

In [ ]:
spark.stop()